In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)

In [ ]:
# Organize Function (equivalent of dplyr's arrange function). From https://stackoverflow.com/questions/75667332/base-analogue-of-arrange-in-pipelines
organize <- function (df, ..., na.last = TRUE, decreasing = FALSE,
                      method = c("auto", "shell", "radix"), drop = FALSE) {
  method <- match.arg(method)
  dots <- eval(substitute(alist(...)))
  exprs <- lapply(dots, FUN = \(e) eval(e, df, parent.frame())) 
  arg_ls <- c(exprs,
              na.last = na.last,
              decreasing = decreasing,
              method = method
              )
  df[do.call("order", arg_ls), , drop = drop]
}

## Reading in Files

In [ ]:
list.files( "/sharedscratch/maff1-group/data" )

In [ ]:
ADNI_merge <- read.csv(file.path("/sharedscratch/maff1-group/data",  "ADNIMERGE_03Feb2025.csv") )
colnames(ADNI_merge)

### IR Data

In [ ]:
ir_all <- read.csv("010_All IR Data")

In [ ]:
ir_bl <- ir_all |>
    subset(VISCODE2 == "bl")

## Merging IR Data w/ ADNI Merge by RID

In [ ]:
tbl1 <- merge(ADNI_merge, ir_bl, by = "RID")
tbl1 |>
    organize(by = RID)

In [ ]:
colnames(tbl1)

In [ ]:
tbl1 <- tbl1 |>
    rename(IR_VISCODE2 = VISCODE2,
           IR_EXAMDATE = EXAMDATE.y)

In [ ]:
# ADNI Merge Cohort w/out any data removed & w/out health data
write.csv(tbl1, "025_ADNI Merge Cohort V1")

In [ ]:
# Checking for NA Values
sum(is.na(tbl1$TG_HDL_Ratio)) 
sum(is.na(tbl1$TYG))
sum(is.na(tbl1$TYG_BMI))
sum(is.na(tbl1$METS_IR))

In [ ]:
# Removing NA Values
tbl2 <- tbl1 |>
  drop_na(TG_HDL_Ratio, TYG, TYG_BMI, METS_IR)
# Checking for NA Values
sum(is.na(tbl2$TG_HDL_Ratio)) 
sum(is.na(tbl2$TYG))
sum(is.na(tbl2$TYG_BMI))
sum(is.na(tbl2$METS_IR))

In [ ]:
nrow(tbl2)

In [ ]:
# Cohort will NA values excluded in IR Categories
write.csv(tbl2, "026_ADNI Merge Cohort V2")

## Adding Health Data

In [ ]:
table(tbl2$ORIGPROT)

### MedHist (Smoking, Alcohol, Drugs, & CVD)

In [ ]:
# Med Hist Contains Health History Data from Participants in ADNI 1, 2, and GO
cmd2 <- "tar -xOzf /sharedscratch/maff1-group/data/ADNI_MedicalHistory.tar.gz MEDHIST_04Jun2025.csv"
medhist <- read.csv( pipe(cmd2) )
medhist |>
    organize(by = RID)

In [ ]:
medhist_short <- medhist[, colnames(medhist) %in% c('RID','PHASE','VISCODE2','VISDATE', 
                                                   'MH4CARD', 'MH14ALCH', 'MH15DRUG', 'MH16SMOK')]
medhist_short <- medhist_short |>
    mutate(SUBSTANCE_AB = if_else(MH14ALCH + MH15DRUG + MH16SMOK >= 1, 1, 0)) |>
    rename(CVD = MH4CARD)
medhist_short |>
    organize(by = RID)

In [ ]:
# Keeping only medical history at screening
medhist_short_sc <- medhist_short |>
    subset(VISCODE2 == "sc")

In [ ]:
# Merging MedHist w/ Cohort
tbl3 <- merge(tbl2, medhist_short_sc, by = "RID")

### Vitals (Hypertension & BMI)

In [ ]:
# Vitals - Hypertension & BMI
cmd <- "tar -xOzf /sharedscratch/maff1-group/data/ADNI_PhysicalNeuro.tar.gz VITALS_04Jun2025.csv"
vitals <- read.csv( pipe(cmd) )

In [ ]:
BP <- vitals[, colnames(vitals) %in% c('RID', 'PHASE', 'VISCODE2', 'VSBPSYS', 'VSBPDIA')]

In [ ]:
# Adding Hypertension Columns. 
    # HPT_S = Hypertension based on systolic measurment >= 140 mmHg
    # HPT_D = Hypertension based on diastolic measurement >= 90 mmHg
    # NICE Guidelines for Hypertension: https://www.nice.org.uk/guidance/ng136
hypertension <- BP |>
    mutate(HPT_S = if_else(VSBPSYS >= 140, 1, 0)) |>
    mutate(HPT_D = if_else(VSBPDIA >= 90, 1, 0)) |>
    mutate(HYPERTENSION = if_else((HPT_D+HPT_S)>=1, 1, 0))

In [ ]:
hypertension_clean <- hypertension[, colnames(hypertension) %in% c('RID', 'PHASE', 'VISCODE2','HYPERTENSION')]

In [ ]:
BMI <- read.csv("015_BMI Data")

In [ ]:
hypertension_clean |>
    organize(by = RID)

In [ ]:
BMI_hypertension <- merge(BMI, hypertension_clean, by = c("RID", "VISCODE2"))
BMI_hypertension <- BMI_hypertension[, colnames(BMI_hypertension) %in% c('RID', 'PHASE','VISCODE2', 'VISDATE','HYPERTENSION', 'BMI')]
BMI_hypertension <- BMI_hypertension |>
    relocate(PHASE, .after = RID)
BMI_hypertension |> 
    organize(by = RID)

In [ ]:
# Keeping Only Baseline
BMI_hypertension_bl <- BMI_hypertension |>
    subset(VISCODE2 == "bl")
# Keepigng Only Screening
BMI_hypertension_sc <- BMI_hypertension |>
    subset(VISCODE2 == "sc")

In [ ]:
# Merging with tbl3 with baseline BMI_hypertentsion data and checking for missing data
tbl4 <- merge(tbl3, BMI_hypertension_bl, by = "RID")
nrow(tbl3)
nrow(tbl4)
sum(is.na(tbl4$BMI)) 
sum(is.na(tbl4$HYPERTENSION))
# Merging with tbl3 with screening BMI_hypertentsion data and checking for missing data
tbl5 <- merge(tbl3, BMI_hypertension_sc, by = "RID")
nrow(tbl3)
nrow(tbl5)
sum(is.na(tbl5$BMI)) 
sum(is.na(tbl5$HYPERTENSION))

In [ ]:
colnames(tbl4)

In [ ]:
# Med history from screening & BMI & Hypertension from baseline
tbl4 <- tbl4 |>
    rename(MEDHIST_VISCODE = VISCODE2.x,
           MEDHIST_DATE = VISDATE.x,
           ALCOHOL = MH14ALCH,
           DRUG_AB = MH15DRUG,
           SMOKING = MH16SMOK, 
           BMIHT_DATE = VISDATE.y)

# Med history from screening & BMI & Hypertension from screening
tbl5 <- tbl5 |>
    rename(MEDHIST_VISCODE = VISCODE2.x,
           MEDHIST_DATE = VISDATE.x,
           ALCOHOL = MH14ALCH,
           DRUG_AB = MH15DRUG,
           SMOKING = MH16SMOK, 
           BMIHT_DATE = VISDATE.y)

### Nightingale (diabetes, TG, HDL, Glucose)

In [ ]:
# Opening nightingale file
cmd2 <- "tar -xOzf /sharedscratch/maff1-group/data/ADNI_Metabolite_NIGHTINGALE.tar.gz ADNINIGHTINGALELONG_05_24_21_04Jun2025.csv"
nightingale <- read.csv( pipe(cmd2) )
nightingale <- nightingale[, colnames(nightingale) %in% c('RID', 'VISCODE','VISCODE2','EXAMDATE', 'TOTAL_TG', 'HDL_C', 'GLUCOSE')]

In [ ]:
# Diabetes is defined by NICE as a fasting plasma glucose above or equal to 7 mmol/L
    # https://cks.nice.org.uk/topics/diabetes-type-2/diagnosis/diagnosis-in-adults/
diabetes <- nightingale |>
    mutate(DIABETES = if_else(GLUCOSE >= 7, 1, 0)) 
diabetes_clean <- diabetes[, colnames(diabetes) %in% c('RID','VISCODE2','EXAMDATE', 'DIABETES', 'TOTAL_TG', 'HDL_C', 'GLUCOSE')]
diabetes_clean

In [ ]:
table(diabetes_clean$VISCODE2)
# No screening data available for metabolic data, only baseline.
diabetes_bl <- diabetes_clean |>
    subset(VISCODE2 == "bl")

In [ ]:
# Merging Metabolic Data (diabetes_clean) with tbl4 (bl) and tbl5 (sc)
tbl6 <- merge(tbl4, diabetes_bl, by = "RID")
tbl7 <- merge(tbl5, diabetes_bl, by = "RID")
nrow(tbl6)
nrow(tbl7)

In [ ]:
tbl6 <- tbl6 |>
    rename(metabolic_VISCODE = VISCODE2,
           metabolic_DATE = EXAMDATE,
           EXAMDATE = EXAMDATE.x)
tbl7 <- tbl7 |>
    rename(metabolic_VISCODE = VISCODE2,
           metabolic_DATE = EXAMDATE,
           EXAMDATE = EXAMDATE.x)

In [ ]:
# Saving cohort with bl metabolic & vitals data + screening medhist data 
    # (w/out cleaning up columns)
write.csv(tbl6, "027_ADNI Merge Cohort V3")

## Cleaning Columns Included

In [ ]:
tbl6 <- tbl6 |>
    rename(CDR = CDRSB,
           CDR_bl = CDRSB_bl,
           IR_DATE = IR_EXAMDATE)

In [ ]:
tbl8 <- tbl6[, colnames(tbl6) %in% c("RID", "COLPROT", "ORIGPROT", "PTID", "SITE", 
                                     "VISCODE", "EXAMDATE", "DX_bl", "AGE", "PTGENDER",
                                     "PTEDUCAT", "PTETHCAT", "PTRACCAT", "PTMARRY", "APOE4",
                                     "ABETA", "TAU", "PTAU",
                                     "CDR", "ADAS11", "ADAS13", "MMSE", "MOCA", "EcogPtTotal", "EcogSPTotal",
                                     "DX", 
                                     "EXAMDATE_bl", "CDR_bl", "ADAS11_bl", "ADAS13_bl", "MMSE_bl", "MOCA_bl", "EcogPtTotal_bl", "EcogSPTotal_bl",
                                     "ABETA_bl", "TAU_bl", "PTAU_bl",
                                     "Years_bl", "Month_bl", "Month", 
                                     "IR_DATE", "TG_HDL_Ratio", "TYG", "TYG_BMI", "METS_IR", 
                                     "MEDHIST_DATE", "CVD", "ALCOHOL", "DRUG_AB", "SMOKING", "SUBSTANCE_AB",
                                     "BMIHT_DATE", "BMI", "HYPERTENSION",
                                     "metabolic_DATE", "HDL_C", "TOTAL_TG", "GLUCOSE", "DIABETES" )]

# Month = Months since baseline
# IR is all tabulated from baseline data
# MEDHIST is all screening data
# BMI, Hypertension, HDLs, TG, GLUCOSE, and DIABETES are all tabulated from baseline data

In [ ]:
# Saving ADNI Merge cohort V4 (cleaned, w/ health data and IR metrics)
write.csv(tbl8, "028_ADNI Merge Cohort V4")